In [1]:
import os

target_folder = "CS6423_knowledge_distillation_project" 
path = os.path.join(os.getcwd(), target_folder)

if not os.getcwd().endswith(target_folder):
    os.chdir(path)

# should match the folder you cloned into
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/cor10/CS6423_knowledge_distillation_project


In [2]:
# Core PyTorch imports
import torch
import torch.nn as nn

# Pruning utilities from PyTorch
import torch.nn.utils.prune as prune

# Your project modules
from modules import datasetPrepper, ImagenetLoader, modelTrainer

# Device setup (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [3]:
# Prepare dataset using your custom datasetPrepper class
# This automatically:
# - Splits into train / validation / test
# - Applies transforms
# - Builds dataloaders

data = datasetPrepper(
    dataframe_path="data/labels.csv",     # CSV containing labels
    image_dir="data/test_images",         # Directory with images
    test_split=0.2,                       # 20% reserved for test/val
    val_test_split=0.5,                   # Split that 20% into val/test
    batch_size=32
).prepare(compute_class_weights=True)     # Helps with class imbalance

print("Number of classes:", len(data.class_names))

Number of classes: 61


In [4]:
# Initialize model loader
loader = ImagenetLoader()

# Load pretrained ResNet50 (teacher model)
teacher = loader.load_radimagenet_resnet50(
    weights_path="./trained_models/resnet50_baseline_gpu_new/resnet50_baseline_gpu_new.pth",
    load_type="load"   # load already trained model
)

# Move teacher to device
teacher = teacher.to(device)

# Set teacher to evaluation mode (VERY IMPORTANT for KD)
teacher.eval()

# Freeze teacher parameters (we NEVER train the teacher)
for param in teacher.parameters():
    param.requires_grad = False

print("Teacher model loaded and frozen.")

Teacher model loaded and frozen.


# UNTESTED BEYOND THIS POINT

In [5]:
# Load a smaller model (student)
loader.load_radimagenet_resnet18(
    weights_path="./trained_models/resnet18_baseline/resnet18_baseline.pth",
    load_type="load"
)

# Freeze most of the backbone to prevent overfitting
# Only train:
# - Fully connected layer
# - Optionally layer4 (higher-level features)
loader.freeze_backbone(finetune_layer4=True)

# Extract the model
student = loader.model

# Move student to device
student = student.to(device)

print("Student model ready.")

Student model ready.


In [6]:
class PWKDLoss(nn.Module):
    """
    Custom loss for Pruning + Knowledge Distillation (PWKD)

    Combines:
    - CrossEntropyLoss (ground truth learning)
    - KL Divergence (teacher guidance)

    This class also signals that it requires input images
    so that the teacher can compute outputs.
    """

    def __init__(self, teacher, temperature=4.0, alpha=0.5):
        super().__init__()

        self.teacher = teacher            # teacher model
        self.temperature = temperature    # softening factor
        self.alpha = alpha                # balance between CE and KD

        self.ce = nn.CrossEntropyLoss()   # standard classification loss
        self.kl = nn.KLDivLoss(reduction="batchmean")  # KD loss

        # Flag to tell trainer we need input images
        self.requires_inputs = True

    def forward(self, student_logits, labels, inputs):
        """
        student_logits: output from student model
        labels: ground truth labels
        inputs: original images (needed for teacher)
        """

        # Get teacher predictions (no gradients!)
        with torch.no_grad():
            teacher_logits = self.teacher(inputs)

        # Standard classification loss
        ce_loss = self.ce(student_logits, labels)

        # Knowledge distillation loss
        kd_loss = self.kl(
            torch.log_softmax(student_logits / self.temperature, dim=1),
            torch.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)

        # Combine both losses
        total_loss = self.alpha * ce_loss + (1 - self.alpha) * kd_loss

        return total_loss

In [6]:
class PWKDLoss(nn.Module):
    """
    Custom loss for Pruning + Knowledge Distillation (PWKD)

    Combines:
    - CrossEntropyLoss (ground truth learning)
    - KL Divergence (teacher guidance)

    This class also signals that it requires input images
    so that the teacher can compute outputs.
    """

    def __init__(self, teacher, temperature=4.0, alpha=0.5):
        super().__init__()

        self.teacher = teacher            # teacher model
        self.temperature = temperature    # softening factor
        self.alpha = alpha                # balance between CE and KD

        self.ce = nn.CrossEntropyLoss()   # standard classification loss
        self.kl = nn.KLDivLoss(reduction="batchmean")  # KD loss

        # Flag to tell trainer we need input images
        self.requires_inputs = True

    def forward(self, student_logits, labels, inputs):
        """
        student_logits: output from student model
        labels: ground truth labels
        inputs: original images (needed for teacher)
        """

        # Get teacher predictions (no gradients!)
        with torch.no_grad():
            teacher_logits = self.teacher(inputs)

        # Standard classification loss
        ce_loss = self.ce(student_logits, labels)

        # Knowledge distillation loss
        kd_loss = self.kl(
            torch.log_softmax(student_logits / self.temperature, dim=1),
            torch.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)

        # Combine both losses
        total_loss = self.alpha * ce_loss + (1 - self.alpha) * kd_loss

        return total_loss

In [7]:
class PWKDLoss(nn.Module):
    """
    Custom loss for Pruning + Knowledge Distillation (PWKD)

    Combines:
    - CrossEntropyLoss (ground truth learning)
    - KL Divergence (teacher guidance)

    This class also signals that it requires input images
    so that the teacher can compute outputs.
    """

    def __init__(self, teacher, temperature=4.0, alpha=0.5):
        super().__init__()

        self.teacher = teacher            # teacher model
        self.temperature = temperature    # softening factor
        self.alpha = alpha                # balance between CE and KD

        self.ce = nn.CrossEntropyLoss()   # standard classification loss
        self.kl = nn.KLDivLoss(reduction="batchmean")  # KD loss

        # Flag to tell trainer we need input images
        self.requires_inputs = True

    def forward(self, student_logits, labels, inputs):
        """
        student_logits: output from student model
        labels: ground truth labels
        inputs: original images (needed for teacher)
        """

        # Get teacher predictions (no gradients!)
        with torch.no_grad():
            teacher_logits = self.teacher(inputs)

        # Standard classification loss
        ce_loss = self.ce(student_logits, labels)

        # Knowledge distillation loss
        kd_loss = self.kl(
            torch.log_softmax(student_logits / self.temperature, dim=1),
            torch.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)

        # Combine both losses
        total_loss = self.alpha * ce_loss + (1 - self.alpha) * kd_loss

        return total_loss

In [8]:
# Create PWKD loss
pwkd_loss = PWKDLoss(
    teacher=teacher,
    temperature=4.0,
    alpha=0.5
)

# Initialize trainer
trainer = modelTrainer(
    model=student,
    data_prep=data,
    device=device,
    learn_rate=1e-4,
    num_epochs=10,
    model_name="pwkd_resnet18"
)

# Prepare training
trainer.prepare_for_training(
    trainable_params=loader.get_trainable_params(),
    loss_fn=pwkd_loss
)

print("Trainer ready.")

Trainer ready.


In [9]:
def apply_pruning(model, amount=0.05):
    """
    Apply unstructured L1 pruning to model weights.

    amount=0.05 means 5% of weights are removed each time.
    """

    for module in model.modules():

        # Only prune Conv and Linear layers
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):

            prune.l1_unstructured(
                module,
                name="weight",
                amount=amount
            )

In [10]:
best_val_f1 = 0

# Loop over epochs
for epoch in range(trainer.num_epochs):

    print(f"\n===== Epoch {epoch+1} =====")

    # Train for one epoch
    train_loss, train_f1 = trainer.train_epoch(epoch)

    # Validate model
    val_loss, val_f1 = trainer.validate()

    # Apply pruning AFTER each epoch (this is key to PWKD)
    apply_pruning(student, amount=0.05)

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        trainer.save_model(current_epoch=epoch + 1)

    # Print metrics
    print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")


===== Epoch 1 =====


Validating: 100%|██████████| 32/32 [00:02<00:00, 13.59batch/s]


Train Loss: 1.4585 | Train F1: 0.6605
Val Loss: 1.5108 | Val F1: 0.4618

===== Epoch 2 =====


Validating: 100%|██████████| 32/32 [00:02<00:00, 13.93batch/s]


Train Loss: 1.2901 | Train F1: 0.6925
Val Loss: 1.4694 | Val F1: 0.4779

===== Epoch 3 =====


...

Train Loss: 1.0005 | Train F1: 0.7554
Val Loss: 1.3563 | Val F1: 0.4884

===== Epoch 9 =====


Validating: 100%|██████████| 32/32 [00:02<00:00, 14.37batch/s]


Train Loss: 0.9798 | Train F1: 0.7668
Val Loss: 1.3579 | Val F1: 0.4884

===== Epoch 10 =====


Validating: 100%|██████████| 32/32 [00:02<00:00, 13.84batch/s]


Train Loss: 0.9685 | Train F1: 0.7719
Val Loss: 1.3142 | Val F1: 0.4998


In [11]:
# Remove pruning reparameterization
# This makes pruning permanent (important for evaluation)

for module in student.modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        try:
            prune.remove(module, "weight")
        except:
            pass

print("Pruning finalized.")

Pruning finalized.


In [14]:
from modules import ModelEvaluator

# Create evaluator
evaluator = ModelEvaluator(
    data_loader=data.test_loader,
    class_names=data.class_names,
    device=device
)

baseline_student = loader.load_radimagenet_resnet18(
    "./trained_models/resnet18_baseline_gpu_new/resnet18_baseline_gpu_new.pth",
    load_type="load"
    )

#compare all models, teacher, initial student architecture, and final PWKD student
evaluator.evaluate_many({
    "Teacher ResNet50": teacher,
    "Pre-PWKD Student": baseline_student,
    "Post-PWKD Student": student
})
# Evaluate student model
# metrics = evaluator.evaluate_single(student, "PWKD Student")

print("\nFinal Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v}")


[Evaluating] Teacher ResNet50...

Warming up Teacher ResNet50...
Running inference...

[Evaluating] Pre-PWKD Student...

Warming up Pre-PWKD Student...
Running inference...

[Evaluating] Post-PWKD Student...

Warming up Post-PWKD Student...
Running inference...

+-------------------+------------+---------------------+-------------------+
| Model             |   F1 Macro |   cuda Latency (ms) |   Model Size (mb) |
+===================+============+=====================+===================+
| Teacher ResNet50  |       0.48 |                0.18 |             90.36 |
+-------------------+------------+---------------------+-------------------+
| Pre-PWKD Student  |       0.4  |                0.11 |             42.79 |
+-------------------+------------+---------------------+-------------------+
| Post-PWKD Student |       0.43 |                0.13 |             42.79 |
+-------------------+------------+---------------------+-------------------+

Final Metrics:
f1_micro: 0.502012072434607


[Evaluating] Teacher ResNet50...

Warming up Teacher ResNet50...
Running inference...

[Evaluating] Post-PWKD Student...

Warming up Post-PWKD Student...
Running inference...

+-------------------+------------+---------------------+-------------------+
| Model             |   F1 Macro |   cuda Latency (ms) |   Model Size (mb) |
+===================+============+=====================+===================+
| PWKD Student      |       0.43 |                0.12 |             42.79 |
+-------------------+------------+---------------------+-------------------+
| Teacher ResNet50  |       0.48 |                0.17 |             90.36 |
+-------------------+------------+---------------------+-------------------+
| Post-PWKD Student |       0.43 |                0.1  |             42.79 |
+-------------------+------------+---------------------+-------------------+
